# 02. Árstruktúra és Piaci Szegmentáció

**Cél:** a lakáspiac belső tagozódásának és az épületfizikai attribútumok (építőanyag, állapot, felszereltség) izolált árazási hatásának feltárása.

---

### 📖 A piac strukturális heterogenitása
A városi ingatlanpiac nem homogén termék piaca, hanem részpiacok halmaza:
1. **Épülettípus:** az iparosított technológia (panel) és a hagyományos falazat eltérő avulási rátával és társadalmi megítéléssel bír.
2. **Műszaki állapot:** a felújított és felújítandó lakások közötti árrés a „rent gap" indikátora.
3. **Kényelmi funkciók:** az erkély, a lift, a parkoló a modern életminőség szűk keresztmetszeteit árazza be.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from ingatlan_tdk._utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu

df = load_szamitott_master()
display(HTML(render_area_header_html(df=df)))
elado = df[df['listing_type'] == 'elado'].copy()
print(f"Elemzett eladó lakások száma: {len(elado)} db.")


### 1. Panel Diszkont és Építőanyag-specifikus Árazás

**📌 Megfigyelések (boxplot-analízis):**
A dobozdiagramok az építési technológiák szerinti fajlagos áreloszlásokat mutatják; a doboz a középső 50%-ot (IQR), a középső vonal a mediánt jelöli. Az építőanyag besorolása **háromértékű**: panel, tégla, illetve egyéb/ismeretlen (a kategóriadummyk alapján nem besorolható hirdetések).

**🔍 Tudományos értelmezés:**
- **Panel diszkont:** az iparosított technológiájú lakások szinte minden urbanizált környezetben rendszerszintű negatív áreltérést mutatnak — az okok az alacsonyabb energetikai hatékonyság, a kötött alaprajz és a magasabb amortizációs kockázat.
- **Szóráskülönbségek:** a panelek dobozai „feszesebbek" (standardizált termék), a téglalakások nagyobb szórása a minőség, életkor és elhelyezkedés diverzitásából fakad.
- **Statisztikai teszt:** a medián-árkülönbség szignifikanciáját **Mann–Whitney U teszttel** ellenőrizzük; a p-érték a diagram annotációjában látható.

In [ ]:
# Háromértékű építőanyag-változó: Panel / Tégla / Egyéb (Ismeretlen)
# (a korábbi "is_panel == 0 → Tégla" besorolás helyett a két dummy külön-külön értelmezve)
elado['Epites_Tipus'] = np.select(
    [elado['is_panel'] == 1, elado['is_tegla'] == 1],
    ['Panel', 'Tégla'],
    default='Egyéb / Ismeretlen'
)

n_panel = int((elado['Epites_Tipus'] == 'Panel').sum())
n_tegla = int((elado['Epites_Tipus'] == 'Tégla').sum())
n_egyeb = int((elado['Epites_Tipus'] == 'Egyéb / Ismeretlen').sum())
med_panel = elado.loc[elado['Epites_Tipus'] == 'Panel', 'nm_ar_huf'].median()
med_tegla = elado.loc[elado['Epites_Tipus'] == 'Tégla', 'nm_ar_huf'].median()

if n_panel > 0 and n_tegla > 0 and pd.notna(med_panel) and med_panel > 0:
    tegla_premium_pct = (med_tegla - med_panel) / med_panel * 100
    premium_txt = f"{'+' if tegla_premium_pct >= 0 else ''}{tegla_premium_pct:.1f}%"
else:
    premium_txt = "— (nincs panel-minta)"

kpi_cards = [
    ("Panel Lakások", f"{n_panel:,} db".replace(',', ' '), f"{n_panel/len(elado)*100:.1f}% kínálat", "#ef4444"),
    ("Tégla Lakások", f"{n_tegla:,} db".replace(',', ' '), f"{n_tegla/len(elado)*100:.1f}% kínálat", "#06b6d4"),
    ("Egyéb / Ismeretlen", f"{n_egyeb:,} db".replace(',', ' '), f"{n_egyeb/len(elado)*100:.1f}% kínálat", "#94a3b8"),
    ("Panel Medián Ár/m²", fmt_huf(med_panel), "Fajlagos ár", "#f87171"),
    ("Tégla Medián Ár/m²", fmt_huf(med_tegla), "Fajlagos ár", "#22d3ee"),
    ("Tégla Prémium (Panelhez képest)", premium_txt, "Medián fajlagos árkülönbözet", "#10b981"),
    ("Prémium Kategória (>80M)", f"{(elado['ar_millio_ft'] > 80).sum()} db", "Felső piaci szegmens", "#8b5cf6")
]
display(HTML(kpi_grid_html(kpi_cards)))


### 2. Kényelmi Funkciók Marginális Prémiuma

**📌 Értékvezérlő paraméterek:**
- **Erkély/terasz:** a sűrű beépítésű környezetben a privát kültér szűkössége miatt mérhető felára van.
- **Lift:** az akadálymentesítés, az elöregedő társadalom és a magasabb emeleti lakások jobb elérhetősége miatt kiemelt értékvezérlő.

**🔍 Statisztikai ellenőrzés:** a lift-hatás medián-különbségét **Mann–Whitney U teszttel** teszteljük; a p-érték a diagram címében látható.

In [ ]:
# Építőanyag-típusok (háromértékű változó) négyzetméterár-eloszlása városrészenként
fig1 = px.violin(
    elado,
    x='varosresz',
    y='nm_ar_huf',
    color='Epites_Tipus',
    box=True,
    points='outliers',
    title='Építőanyag-típusok (Panel / Tégla / Egyéb) négyzetméterár-eloszlása városrészenként',
    labels={'varosresz': 'Városrész', 'nm_ar_huf': 'Ár / m² (HUF)', 'Epites_Tipus': 'Épülettípus'},
    color_discrete_map={'Panel': '#ef4444', 'Tégla': '#06b6d4', 'Egyéb / Ismeretlen': '#94a3b8'},
    category_orders={'Epites_Tipus': ['Panel', 'Tégla', 'Egyéb / Ismeretlen']},
    template=PLOTLY_TEMPLATE
)

# Mann–Whitney U: panel vs. nem-panel medián-összehasonlítás (p-érték az annotációban)
panel_ar = elado.loc[elado['Epites_Tipus'] == 'Panel', 'nm_ar_huf'].dropna()
nem_panel_ar = elado.loc[elado['Epites_Tipus'] != 'Panel', 'nm_ar_huf'].dropna()
if len(panel_ar) >= 5 and len(nem_panel_ar) >= 5:
    mw_panel = mannwhitneyu(panel_ar, nem_panel_ar, alternative='two-sided')
    mw_panel_txt = (
        f"Mann–Whitney U (panel vs. nem-panel): p = {mw_panel.pvalue:.4f} "
        f"(n = {len(panel_ar)} vs. {len(nem_panel_ar)})"
    )
else:
    mw_panel_txt = "Mann–Whitney U (panel vs. nem-panel): nem futtatható — túl kevés panel-megfigyelés"
fig1.update_layout(
    xaxis_tickangle=-30,
    height=480,
    margin=dict(t=110),
    annotations=[dict(
        x=0.5, y=1.12, xref='paper', yref='paper',
        text=mw_panel_txt, showarrow=False,
        font=dict(size=12, color='#334155')
    )]
)
fig1.show()
print(mw_panel_txt)

# Lift-hatás: has_lift medián-összehasonlítás Mann–Whitney U teszttel (p-érték a címben)
lift_df = elado.dropna(subset=['has_lift', 'nm_ar_huf']).copy()
lift_df['Lift'] = lift_df['has_lift'].map({1: 'Van lift', 0: 'Nincs lift'})
lift_ar = lift_df.loc[lift_df['has_lift'] == 1, 'nm_ar_huf']
nolift_ar = lift_df.loc[lift_df['has_lift'] == 0, 'nm_ar_huf']
med_lift = lift_ar.median()
med_nolift = nolift_ar.median()
if len(lift_ar) >= 5 and len(nolift_ar) >= 5 and med_nolift > 0:
    mw_lift = mannwhitneyu(lift_ar, nolift_ar, alternative='two-sided')
    lift_diff_pct = (med_lift - med_nolift) / med_nolift * 100
    lift_title = (
        f"Lift-hatás a fajlagos árakban — medián: {fmt_huf(med_lift)} vs. {fmt_huf(med_nolift)} "
        f"({lift_diff_pct:+.1f}%) — Mann–Whitney U p = {mw_lift.pvalue:.4f}"
    )
    lift_txt = (
        f"Mann–Whitney U (lift vs. nem lift): p = {mw_lift.pvalue:.4f} "
        f"(n = {len(lift_ar)} vs. {len(nolift_ar)})"
    )
else:
    lift_title = "Lift-hatás a fajlagos árakban — Mann–Whitney U nem futtatható (túl kevés megfigyelés)"
    lift_txt = lift_title
fig_lift = px.violin(
    lift_df,
    x='Lift',
    y='nm_ar_huf',
    color='Lift',
    box=True,
    points='outliers',
    title=lift_title,
    labels={'Lift': 'Lift megléte', 'nm_ar_huf': 'Ár / m² (HUF)'},
    color_discrete_map={'Van lift': '#059669', 'Nincs lift': '#94a3b8'},
    category_orders={'Lift': ['Nincs lift', 'Van lift']},
    template=PLOTLY_TEMPLATE
)
fig_lift.update_layout(height=430, showlegend=False)
fig_lift.show()
print(lift_txt)


### 3. Állapot-alapú Értékcsökkenés (Amortizációs Görbe)

**📌 A műszaki állapot árazása:**
A grafikon az állapotkategóriák (felújítandó → új építésű) szerinti medián négyzetméterárakat ábrázolja.

**🔍 Rent Gap (különbözeti járadék) elmélet:**
A felújított és felújítandó lakások közötti fajlagos árkülönbség elméletileg a felújítás átlagos négyzetméter-költségét plusz a kockázati prémiumot tartalmazza. Ha az árrés ezt érdemben meghaladja, az **térbeli arbitrázslehetőséget** jelez — a tőke beáramlása (dzsentrifikáció) várható a szegmensben.

In [ ]:
# Szobaszám-kategória: a kanonikus adatoszlop (szobaszam_kategoria) használata
# (a hiányzó értékek a pivot-összegzésből kiesnek)

# Méretkategóriák: a kanonikus, kvantilis-alapú MERET_BINS / MERET_LABELS (csak leíró ábrákhoz)
elado['meret_kategoria'] = pd.cut(
    elado['alapterulet_nm'],
    bins=MERET_BINS,
    labels=MERET_LABELS,
    include_lowest=True
)

# Kereszttábla aggregáció
pivot_nmar = elado.pivot_table(
    index='meret_kategoria',
    columns='szobaszam_kategoria',
    values='nm_ar_huf',
    aggfunc='median',
    observed=False
)

fig2 = px.imshow(
    pivot_nmar,
    text_auto='.0f',
    color_continuous_scale='Blues',
    title='Medián Négyzetméterár (Ft/m²) Méret és Szobaszám Mátrixban',
    labels={'x': 'Szobaszám Kategória', 'y': 'Alapterület Kategória', 'color': 'Ár / m²'},
    template=PLOTLY_TEMPLATE
)
fig2.update_layout(height=420)
fig2.show()

# Emeleti árazási hatás — hiányzó-dummyval:
# a NaN (ismeretlen) dummyk NEM "Közbenső emelet"-ként osztályozódnak, hanem külön kategóriát kapnak
def emelet_pozicio(r):
    if r['is_foldszint'] == 1:
        return 'Földszint'
    if r['is_zaroszint'] == 1:
        return 'Zárószint'
    if pd.isna(r['is_foldszint']) or pd.isna(r['is_zaroszint']):
        return 'Ismeretlen (nincs adat)'
    return 'Közbenső emelet'

elado['Pozicio'] = elado[['is_foldszint', 'is_zaroszint']].apply(emelet_pozicio, axis=1)
pozicio_sorrend = ['Földszint', 'Közbenső emelet', 'Zárószint', 'Ismeretlen (nincs adat)']

emelet_stat = elado.groupby('Pozicio')['nm_ar_huf'].agg(median='median', count='size').reset_index()

# Mann–Whitney U: földszint és zárószint diszkontja a közbenső emeletekhez képest (p-érték a címben)
fold_ar = elado.loc[elado['Pozicio'] == 'Földszint', 'nm_ar_huf'].dropna()
koz_ar = elado.loc[elado['Pozicio'] == 'Közbenső emelet', 'nm_ar_huf'].dropna()
zar_ar = elado.loc[elado['Pozicio'] == 'Zárószint', 'nm_ar_huf'].dropna()
mw_annot = []
for nev, a, b in [('Földszint vs. Közbenső', fold_ar, koz_ar), ('Zárószint vs. Közbenső', zar_ar, koz_ar)]:
    if len(a) >= 5 and len(b) >= 5:
        mw = mannwhitneyu(a, b, alternative='two-sided')
        mw_annot.append(f"{nev}: Mann–Whitney U p = {mw.pvalue:.4f} (n = {len(a)} vs. {len(b)})")
    else:
        mw_annot.append(f"{nev}: Mann–Whitney U nem futtatható (túl kevés megfigyelés)")
mw_title_suffix = '<br><sup>' + '; '.join(mw_annot) + '</sup>'

fig3 = px.bar(
    emelet_stat,
    x='Pozicio',
    y='median',
    color='Pozicio',
    text='count',
    title='Földszint és Zárószint Árdiszkontja a Közbenső Emeletekhez Képest' + mw_title_suffix,
    labels={'Pozicio': 'Emeleti Elhelyezkedés', 'median': 'Medián Ár / m² (HUF)', 'count': 'N'},
    category_orders={'Pozicio': pozicio_sorrend},
    template=PLOTLY_TEMPLATE
)
fig3.update_layout(height=430, showlegend=False)
fig3.show()
print('; '.join(mw_annot))


### 4. Interaktív Városrész- és Szegmens Laboratórium

**📌 Mire használható a vezérlőfelület?**
A lenyíló menükkel városrészenként szűrheti a panel, tégla és egyéb/ismeretlen lakások megoszlását, valamint az állapot- és szobaszám-kategóriákat.
- Az építőanyag-megoszlás városrészenként jelentősen eltér — ez is magyarázza a városrészek közötti általános árszint-különbséget!


In [ ]:
# Interaktív Piaci Szegmentációs Elemző (Plotly updatemenus)
szegmens_vars = [
    ('Epites_Tipus', '1. Épülettípus (Panel vs. Tégla)', '#2563eb'),
    ('allapot_kod', '2. Műszaki Állapot Kategória', '#059669'),
    ('szobaszam_kategoria', '3. Szobaszám Szerinti Kategória', '#7c3aed')
]

fig_szeg = go.Figure()
buttons = []

for i, (col, label, colr) in enumerate(szegmens_vars):
    if col in elado.columns:
        agg = elado.groupby(col, observed=True)['nm_ar_huf'].agg(['count', 'median', 'mean']).reset_index()
        sub_fig = px.bar(
            agg, x=col, y='median', text='count',
            labels={'median': 'Medián Ár/m² (Ft)', col: label}
        )
        tr = sub_fig.data[0]
        tr.visible = (i == 0)
        tr.marker.color = colr
        tr.name = label
        fig_szeg.add_trace(tr)
        
        vis = [j == i for j in range(len(szegmens_vars))]
        buttons.append(dict(
            label=label,
            method='update',
            args=[{'visible': vis}, {'title': f'Medián Négyzetméterár {label} szerint (N={len(elado)})', 'xaxis': {'title': label}}]
        ))

fig_szeg.update_layout(
    title=f'Medián Négyzetméterár {szegmens_vars[0][1]} szerint (N={len(elado)})',
    xaxis_title=szegmens_vars[0][1],
    yaxis_title='Medián Fajlagos Ár (Ft/m²)',
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        direction='down',
        x=0.01, y=0.99, xanchor='left', yanchor='top',
        bgcolor='white', bordercolor='#cbd5e1'
    )],
    template=PLOTLY_TEMPLATE,
    height=450
)
fig_szeg.show()